# 10_powerbi_export: summary tables for the Power BI dashboard
Writes five CSV files to `reports/powerbi/` for the service desk dashboard. Inputs: `itsm_clean.pkl` (03), `track1_test.pkl` (05) and the saved Track 1 model (06). Aggregates only: no ticket IDs, CI names or other asset codes leave this notebook.

| File | One row per | Used for |
|---|---|---|
| `ticket_summary.csv` | date x hour x category x asset type x priority group | Overview and high-priority risk pages |
| `alert_outcomes.csv` | date x asset type x outcome | Model alerts page |
| `daily_forecast.csv` | date x series | Forecast page (actuals and SARIMA forecast) |
| `quarterly_forecast.csv` | quarter x series | Forecast table with a P10 to P90 range |
| `annual_forecast.csv` | series | Annual forecast cards and table |

The business summary page also uses `objective_summary.csv` and `future_improvements.csv`. They are written by hand (findings and next steps, not calculations), so they are kept in `reports/powerbi/` and not created here.

In [1]:
import os
import sys

import joblib
import numpy as np
import pandas as pd

sys.path.append("../src")
from project_config import CUT, HIGH, START, TEST_START

OUT = "../reports/powerbi"
os.makedirs(OUT, exist_ok=True)

df = pd.read_pickle("../data/processed/itsm_clean.pkl")
w = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT)].copy()
print("Modelling window:", len(w), "tickets,", w["Open_Time"].min().date(), "to", w["Open_Time"].max().date())

Modelling window: 42304 tickets, 2013-10-01 to 2014-03-16


## 1. Ticket summary (Overview and High-priority pages)

In [2]:
w["Date"] = w["Open_Time"].dt.normalize()
w["Hour"] = w["Open_Time"].dt.hour
w["Weekday"] = w["Open_Time"].dt.day_name()
w["Weekday_No"] = w["Open_Time"].dt.day_of_week + 1         # 1 = Monday, for sorting in Power BI
w["Priority_Group"] = np.select(
    [w["Priority"].isin(HIGH), w["Priority"].notna()],
    ["High (P1-P2)", "Normal (P3-P5)"],
    default="Unlabelled")
w["Reassigned"] = (w["No_of_Reassignments"].fillna(0) > 0).astype(int)
w["CI_Cat"] = w["CI_Cat"].fillna("Unknown")

ticket_summary = (w.groupby(["Date", "Weekday", "Weekday_No", "Hour", "Category", "CI_Cat", "Priority_Group"])
                    .agg(Tickets=("Incident_ID", "size"), Reassigned_Tickets=("Reassigned", "sum"))
                    .reset_index()
                    .rename(columns={"CI_Cat": "Asset_Type"}))
ticket_summary.to_csv(f"{OUT}/ticket_summary.csv", index=False)
print("ticket_summary.csv:", len(ticket_summary), "rows,", ticket_summary["Tickets"].sum(), "tickets")
print(ticket_summary.groupby("Priority_Group")["Tickets"].sum())

ticket_summary.csv: 8733 rows, 42304 tickets
Priority_Group
High (P1-P2)        656
Normal (P3-P5)    40418
Unlabelled         1230
Name: Tickets, dtype: int64


## 2. Model alerts on the test period (Model alerts page)
The saved Track 1 model and threshold from `06_track1_models`, applied to the test period (1 February to 16 March 2014). The totals should match Table 7a: 67 caught, 48 missed, 158 false alarms.

In [3]:
saved = joblib.load("../models/track1_priority_lr.joblib")
test = pd.read_pickle("../data/processed/track1_test.pkl")
proba = saved["model"].predict_proba(test[saved["features"]].astype(str))[:, 1]
alert = proba >= saved["threshold"]
high = test["High"].to_numpy() == 1

test["Outcome"] = np.select([alert & high, ~alert & high, alert & ~high],
                            ["Caught", "Missed", "False alarm"], default="Correctly not flagged")
test["Date"] = test["Open_Time"].dt.normalize()
alert_outcomes = (test.groupby(["Date", "CI_Cat", "Outcome"]).size()
                      .rename("Tickets").reset_index()
                      .rename(columns={"CI_Cat": "Asset_Type"}))
alert_outcomes.to_csv(f"{OUT}/alert_outcomes.csv", index=False)
print("alert_outcomes.csv:", len(alert_outcomes), "rows")
print(alert_outcomes.groupby("Outcome")["Tickets"].sum())

alert_outcomes.csv: 396 rows
Outcome
Caught                     67
Correctly not flagged    9956
False alarm               158
Missed                     48
Name: Tickets, dtype: int64


## 3. SARIMA forecasts (Forecast page)
Refits the order chosen in `07_track2_volume`, SARIMA(1,0,0)(1,1,1,7), on daily counts for the whole window, for all tickets and for incidents only, and forecasts to 31 March 2015. The quarterly and annual ranges are the 10th to 90th percentile of 500 simulated paths; the annual range comes from each path's 12-month total. Compare the quarterly and annual totals with Tables 15 and 18 in notebook 07 and the README (about 75,000 incidents and 93,000 to 94,000 tickets a year); small differences come from the random simulations.

In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

ORDER, SEASONAL = (1, 0, 0), (1, 1, 1, 7)
series = {
    "All tickets": w.set_index("Open_Time").resample("D").size().astype(float),
    "Incidents": w[w["Category"] == "incident"].set_index("Open_Time").resample("D").size().astype(float),
}

daily_rows, quarter_rows, annual_rows = [], [], []
np.random.seed(42)
for name, daily in series.items():
    res = SARIMAX(daily, order=ORDER, seasonal_order=SEASONAL).fit(disp=False)
    h = (pd.Timestamp("2015-03-31") - daily.index[-1]).days
    idx = pd.date_range(daily.index[-1] + pd.Timedelta(days=1), periods=h, freq="D")
    point = pd.Series(np.clip(np.asarray(res.forecast(h)), 0, None), index=idx)
    sims = pd.DataFrame(np.asarray(res.simulate(nsimulations=h, repetitions=500, anchor="end")).reshape(h, -1), index=idx)

    daily_rows.append(pd.DataFrame({"Date": daily.index, "Series": name, "Type": "Actual", "Tickets": daily.values}))
    daily_rows.append(pd.DataFrame({"Date": idx, "Series": name, "Type": "Forecast", "Tickets": point.round(1).values}))

    q_point = point.resample("QS").sum().loc["2014-04-01":]
    q_sims = sims.resample("QS").sum().loc["2014-04-01":]
    quarter_rows.append(pd.DataFrame({
        "Quarter_Start": q_point.index, "Series": name,
        "Forecast": q_point.round().values,
        "Low_P10": q_sims.quantile(0.10, axis=1).round().values,
        "High_P90": q_sims.quantile(0.90, axis=1).round().values}))

    # Annual range from each simulated path's 12-month total (quarterly P10 and P90 cannot be added up)
    a_sims = q_sims.sum(axis=0)
    annual_rows.append({"Series": name, "Period": "Next 12 months (Apr 2014 to Mar 2015)",
                        "Forecast": round(q_point.sum()),
                        "Low_P10": round(a_sims.quantile(0.10)),
                        "High_P90": round(a_sims.quantile(0.90))})

daily_forecast = pd.concat(daily_rows, ignore_index=True)
quarterly_forecast = pd.concat(quarter_rows, ignore_index=True)
quarterly_forecast["Quarter"] = ("Q" + quarterly_forecast["Quarter_Start"].dt.quarter.astype(str)
                                 + " " + quarterly_forecast["Quarter_Start"].dt.year.astype(str))
daily_forecast.to_csv(f"{OUT}/daily_forecast.csv", index=False)
quarterly_forecast.to_csv(f"{OUT}/quarterly_forecast.csv", index=False)
annual_forecast = pd.DataFrame(annual_rows)
annual_forecast.to_csv(f"{OUT}/annual_forecast.csv", index=False)
print(annual_forecast)
print(quarterly_forecast)

        Series                                 Period  Forecast  Low_P10  \
0  All tickets  Next 12 months (Apr 2014 to Mar 2015)     93115    87159   
1    Incidents  Next 12 months (Apr 2014 to Mar 2015)     75164    70908   

   High_P90  
0     98364  
1     79407  
  Quarter_Start       Series  Forecast  Low_P10  High_P90  Quarter
0    2014-04-01  All tickets   23181.0  21322.0   25002.0  Q2 2014
1    2014-07-01  All tickets   23569.0  21530.0   25370.0  Q3 2014
2    2014-10-01  All tickets   23506.0  21664.0   25497.0  Q4 2014
3    2015-01-01  All tickets   22859.0  20621.0   24701.0  Q1 2015
4    2014-04-01    Incidents   18713.0  17189.0   20271.0  Q2 2014
5    2014-07-01    Incidents   19025.0  17374.0   20442.0  Q3 2014
6    2014-10-01    Incidents   18971.0  17412.0   20590.0  Q4 2014
7    2015-01-01    Incidents   18455.0  16974.0   19782.0  Q1 2015


### Findings
- Five CSV files are in `reports/powerbi/`. They hold counts and forecasts only, no ticket rows or asset codes.
- Check that the alert totals match Table 7a in `06` and the quarterly forecasts match `07` before building the dashboard.